完整的 MinGPT 函数，RoPE

理论部分可以看这里：https://blog.csdn.net/qq_52511048/article/details/148852676?spm=1001.2014.3001.5506

在多头注意力上进行修改。相比多头注意力，新增：

theta、freqs、sin、cos等部分

MultiHeadCausalSelfAttention中的ROPE旋转操作

删掉：pos_embedding_matrix + LearnablePositionalEncoding（RoPE 替代）

In [27]:
import numpy as np
import torch
import math
import torch.nn.functional as F

T_max=128 #预设的最大序列长度
D=4 #隐藏维度
V=26 #词表大小
H=2 #都头注意力的头数，需要满足D%H==0
char_to_idx={} #字符到整数索引的映射,形状：{V}（V 是字符表大小）
idx_to_char={} #整数索引回字符的映射,形状：{V}


Dh = D // H  # 每个头的维度，必须是偶数
theta=1.0/(10000**(torch.arange(0,Dh,2)/Dh)) #预计算频率：theta = 10000^(-2i/Dh)，i是头维度索引，取值0到Dh//2
pos=torch.arange(T_max) #生成位置索引 [0,1,2,...,T_max-1]
freqs=torch.outer(pos, theta) #外积得到 位置×频率 的角度矩阵 [T_max, Dh//2]
cos=torch.cos(freqs).repeat(1, 2) #扩展为[T_max,Dh]，两两重复，匹配Q/K的维度
sin=torch.sin(freqs).repeat(1, 2)
cos=cos.detach()
sin=sin.detach()


for i in range(26):
    char_to_idx[chr(ord('a')+i)]=i
    idx_to_char[i]=chr(ord('a')+i)




N_LAYERS=2 #堆叠层数

#改为分层初始化列表
Wq_list,Wk_list,Wv_list,Wo_list=[],[],[],[]
gamma1_list, beta1_list = [], []  #每个 Block 第一个 LayerNorm 的参数
gamma2_list, beta2_list = [], []  #每个 Block 第二个 LayerNorm 的参数
W1_list,b1_list,W2_list,b2_list=[],[],[],[]

for _ in range(N_LAYERS):
    #每层独立初始化 Attention 参数
    Wq=torch.randn(D, D, requires_grad=True) #每个形状：[D, D]。没训练前先随机
    Wk=torch.randn(D, D, requires_grad=True)
    Wv=torch.randn(D, D, requires_grad=True)
    Wo=torch.randn(D, D, requires_grad=True) #多头输出投影矩阵
    torch.nn.init.normal_(Wq, mean=0.0, std=0.02)
    torch.nn.init.normal_(Wk, mean=0.0, std=0.02)
    torch.nn.init.normal_(Wv, mean=0.0, std=0.02)
    torch.nn.init.normal_(Wo, mean=0.0, std=0.02)
    Wq_list.append(Wq)
    Wk_list.append(Wk)
    Wv_list.append(Wv)
    Wo_list.append(Wo)

    #每层独立初始化两个LayerNorm参数
    gamma1,beta1=torch.ones(D, requires_grad=True), torch.zeros(D, requires_grad=True)
    gamma2,beta2=torch.ones(D, requires_grad=True), torch.zeros(D, requires_grad=True)
    gamma1_list.append(gamma1)
    beta1_list.append(beta1)
    gamma2_list.append(gamma2)
    beta2_list.append(beta2)

    #每层独立初始化 FFN 参数
    W1 = torch.randn(D, 4*D, requires_grad=True)
    b1 = torch.randn(4*D, requires_grad=True)
    W2 = torch.randn(4*D, D, requires_grad=True)
    b2 = torch.randn(D, requires_grad=True)
    torch.nn.init.normal_(W1, mean=0.0, std=0.02)
    torch.nn.init.normal_(b1, mean=0.0, std=0.02)
    torch.nn.init.normal_(W2, mean=0.0, std=0.02)
    torch.nn.init.normal_(b2, mean=0.0, std=0.02)
    W1_list.append(W1)
    b1_list.append(b1)
    W2_list.append(W2)
    b2_list.append(b2)

In [28]:
embedding_matrix=torch.empty(26,D,requires_grad=True) #向量化，先随便填的。
torch.nn.init.normal_(embedding_matrix, mean=0.0, std=0.02)

def TokenEmbedding(token_ids): #输入：token_ids [T] → 输出：x [T, D]（D 是隐藏维度）。或者输入：token_ids [B, T] → 输出：x [B, T, D]。高级索引机制，就是好用。另外这里没有包含注意力机制的Encoder Block。这个代码是decoder-only的。
    return embedding_matrix[token_ids] #向量化。注意不能写成torch.tensor(embedding_matrix[token_ids])，这是因为embedding_matrix[token_ids] 本身已经是一个 tensor 了，再用 torch.tensor() 包裹它，相当于创建了一个全新的、不带梯度历史的 tensor。

In [29]:
def MultiHeadCausalSelfAttention(X, Wq, Wk, Wv, Wo): #输入：X [B, T, D] → 输出：attn_output [B, T, D]
    B, T, D=X.shape
    Dh=D//H #每个头的维度

    Q=torch.matmul(X,Wq) #直接乘是没问题的，pytorch支持。
    K=torch.matmul(X,Wk)
    V=torch.matmul(X,Wv)
    Q=Q.view(B, T, H, Dh).permute(0, 2, 1, 3)#这一步是拆分多头，从 [B, T, D] 拆成 [B, T, H, Dh]，再把头维度 H 提前到 [B, H, T, Dh]。.permute(0, 2, 1, 3)：4 维张量中，保持第 0、3 维不变，交换第 1 维和第 2 维的位置。
    K=K.view(B, T, H, Dh).permute(0, 2, 1, 3)
    V=V.view(B, T, H, Dh).permute(0, 2, 1, 3)

    #RoPE旋转操作    
    cos_T=cos[:T] #截取当前序列长度的 cos/sin [T, Dh]
    sin_T=sin[:T]
    cos_T=cos_T.unsqueeze(0).unsqueeze(0) #[T, Dh] --> [1, 1, T, Dh]
    sin_T=sin_T.unsqueeze(0).unsqueeze(0)

    #两两分组旋转核心公式：对向量 (x0, x1) 旋转：(x0*cos - x1*sin,  x0*sin + x1*cos)=(x0*cos, x1*cos)+(-x1*sin, x0*sin)
    def rotate(x):
        #步骤1：把最后一维 Dh 拆成 两两一组 [B,H,T,Dh] → [B,H,T,Dh//2,2]
        x_split=x.reshape(*x.shape[:-1], -1, 2)
        #步骤2：提取偶数维度、奇数维度
        x_even=x_split[..., 0] #第0,2,4...维
        x_odd=x_split[..., 1] #第1,3,5...维
        #步骤3：构造旋转后的配对张量，注意关键：奇数维取负
        x_rot=torch.stack([-x_odd, x_even], dim=-1) #[B,H,T,Dh//2,2]
        #步骤4：还原回原维度 [B,H,T,Dh]
        x_rot=x_rot.reshape(x.shape)
        #步骤5：执行旋转公式
        return x * cos_T + x_rot * sin_T

    Q=rotate(Q) #V不旋转
    K=rotate(K)
    #RoPE旋转操作 结束    

    S=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(Dh) #原始注意力分数 [B, H, T, T]
    

    M=torch.zeros((T,T),dtype=torch.float32)
    rows,cols=torch.triu_indices(T,T,offset=1) #矩阵的上三角，offset=1表示主对角右移1
    M[rows,cols]=-torch.inf #将上三角位置设为负无穷
    S=S+M #注意力掩码处理后的分数[B, H, T, T]

    exp_S=torch.exp(S)
    A=exp_S/torch.sum(exp_S,axis=-1,keepdims=True) #注意力权重[B, H, T, T]

    O=torch.matmul(A,V) #计算投影，[B, H, T, Dh]
    O=O.permute(0, 2, 1, 3).contiguous().view(B,T,D)#这一步是合并多头，从 [B, H, T, Dh] 还原回 [B, T, D]。.contiguous() 的核心作用是确保张量在内存中以连续（contiguous）的方式存储

    O = torch.matmul(O, Wo) #多头特有的部分，融合各个头的信息。

    '''
    print(Q)
    print(K)
    print(V)
    print(S)
    print(M)
    print(A)
    '''
    return O

In [30]:
def LayerNorm(X, gamma, beta): #层归一化
    mu=torch.mean(X,dim=-1,keepdims=True)
    sigma=torch.var(X,axis=-1,keepdims=True)
    normalized_X=(X-mu)/torch.sqrt(sigma+1e-5)
    return gamma*normalized_X+beta #数值乘法和加法，矩阵乘法是torch.matmul。用了gamma和beta这个Loss从0.3726干到0.0643了，推测是不加beta一堆负值过不了gelu。如果模型觉得 “不归一化更好”，它可以通过学习近似恢复出归一化前的原始特征。

In [31]:
def FeedForward(X, W1, b1, W2, b2):
    X=torch.matmul(X,W1)+b1
    X=F.gelu(X)
    X=torch.matmul(X,W2)+b2
    return X

In [32]:
def TransformerBlock(X, Wq, Wk, Wv, Wo, gamma1, beta1, gamma2, beta2, W1, b1, W2, b2): #输入 / 输出：[B, T, D]。问：归一化之后token和token之间的相对大小不就变了吗，还怎么区分token和token？答：归一化之后token向量方向不变，长度变了。长度是一维信息，在方向里多加一维就效果一样了。
    X = X + MultiHeadCausalSelfAttention(LayerNorm(X, gamma1, beta1), Wq, Wk, Wv, Wo)
    X = X + FeedForward(LayerNorm(X, gamma2, beta2), W1, b1, W2, b2)
    return X

In [33]:
gamma_final = torch.ones(D, requires_grad=True)
beta_final = torch.zeros(D, requires_grad=True)

def FinalLayerNorm(X):
    return LayerNorm(X, gamma_final, beta_final)

In [34]:
W_head=torch.randn(D,V,requires_grad=True)
b_head=torch.randn(V,requires_grad=True)

torch.nn.init.normal_(W_head, mean=0.0, std=0.02)
torch.nn.init.normal_(b_head, mean=0.0, std=0.02)

def LinearHead(X): #线性层，输入 [B, T, D] → 输出 [B, T, V]，参数 W_head [D, V], b_head [V]。
    X=torch.matmul(X,W_head)+b_head
    return X

In [35]:
def miniGPT(token_ids): #设计的时候要同时能适用于[B,T]和[T]
    X = TokenEmbedding(token_ids) 

    for i in range(N_LAYERS):
        X = TransformerBlock(
            X,
            Wq_list[i], Wk_list[i], Wv_list[i], Wo_list[i],
            gamma1_list[i], beta1_list[i],
            gamma2_list[i], beta2_list[i],
            W1_list[i], b1_list[i], W2_list[i], b2_list[i]
        )

    X = FinalLayerNorm(X)

    logits = LinearHead(X)

    return logits

In [36]:
all_params = [
    embedding_matrix,
    gamma_final, beta_final,
    W_head, b_head
]

for i in range(N_LAYERS):
    all_params.extend([
        Wq_list[i], Wk_list[i], Wv_list[i], Wo_list[i],
        gamma1_list[i], beta1_list[i],
        gamma2_list[i], beta2_list[i],
        W1_list[i], b1_list[i], W2_list[i], b2_list[i]
    ])

In [37]:
def train(train_text):
    lr=1e-3
    epochs=1000
    seq_len=4

    optimizer=torch.optim.AdamW(all_params,lr=lr) #优化器
    

    X_train=[]
    Y_train=[]
    for i in range(len(train_text)-seq_len):
        X_train.append([char_to_idx[c] for c in train_text[i:i+seq_len]])
        Y_train.append([char_to_idx[c] for c in train_text[i+1:i+seq_len+1]])

    X_train=torch.tensor(X_train)
    Y_train=torch.tensor(Y_train)

    for epoch in range(epochs):
        logits=miniGPT(X_train)
        #logits需要展平为[B*T, V]，Y展平为[B*T]
        loss=F.cross_entropy(logits.reshape(-1,V), Y_train.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch%50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [38]:
def generate(prompt_str,max_new_tokens=10):
    input_ids=torch.tensor([[char_to_idx[c] for c in prompt_str]]) #input_ids是[B, T]

    for _ in range(max_new_tokens):
        logits=miniGPT(input_ids)
        next_id=torch.argmax(logits[:,-1,:],dim=-1,keepdim=True) #logits[:,-1,:]是每个batch最后一个token的概率分布

        input_ids=torch.cat((input_ids,next_id),dim=-1)

    res=[''.join([idx_to_char[i.item()] for i in input_ids[0,:]])]
    return res

In [39]:
if __name__ == "__main__":
    train("abcabcabcabcabcabcabcabcab")
    print("生成结果：", generate("a", max_new_tokens=10))

Epoch 0, Loss: 3.2475
Epoch 50, Loss: 2.8448
Epoch 100, Loss: 2.3719
Epoch 150, Loss: 1.9185
Epoch 200, Loss: 1.5866
Epoch 250, Loss: 1.3877
Epoch 300, Loss: 1.2629
Epoch 350, Loss: 1.1312
Epoch 400, Loss: 0.9630
Epoch 450, Loss: 0.7800
Epoch 500, Loss: 0.6008
Epoch 550, Loss: 0.4461
Epoch 600, Loss: 0.3282
Epoch 650, Loss: 0.2446
Epoch 700, Loss: 0.1867
Epoch 750, Loss: 0.1462
Epoch 800, Loss: 0.1173
Epoch 850, Loss: 0.0961
Epoch 900, Loss: 0.0802
Epoch 950, Loss: 0.0680
生成结果： ['abcabcabcab']
